<a href="https://colab.research.google.com/github/saraxfl/Notebook/blob/main/Collab2_Bloque2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import io
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor


def seleccionar_archivo_local():
    """Muestra una ventana para elegir un archivo CSV local o usa fallback en Colab/Jupyter."""
    try:
        import tkinter as tk
        from tkinter import filedialog

        root = tk.Tk()
        root.withdraw()  # Oculta la ventana principal de Tkinter
        root.attributes("-topmost", True)  # Fuerza a la ventana a salir al frente
        path = filedialog.askopenfilename(
            title="Selecciona el archivo de contaminantes (CSV)",
            filetypes=[("Archivos CSV", "*.csv"), ("Todos los archivos", "*.*")],
        )
        return path
    except Exception:
        # Si se ejecuta en un entorno sin GUI (como un servidor remoto o Colab sin Tkinter)
        from google.colab import files

        print("Selecciona el archivo CSV usando el botón de carga:")
        subidos = files.upload()
        if subidos:
            return next(iter(subidos))
        return None

ruta_archivo = seleccionar_archivo_local()

if ruta_archivo:
    datos = pd.read_csv(ruta_archivo, skiprows=9)
    print(f"\nArchivo cargado correctamente: {ruta_archivo}")
    print("Dimensiones de la matriz original:", datos.shape)
    print("\nEstructura de columnas:")
    print(datos.columns.tolist())
    display(datos.head())
else:
    print("No se seleccionó ningún archivo.")


# Estandarizar los registros temporales a objetos de fecha de pandas
datos["date_dt"] = pd.to_datetime(datos["date"], errors="coerce")

if "hour" in datos.columns:
    datos["hora"] = pd.to_numeric(datos["hour"], errors="coerce")
    datos["timestamp"] = datos["date_dt"].dt.normalize() + pd.to_timedelta(
        datos["hora"], unit="h"
    )
else:
    datos["hora"] = datos["date_dt"].dt.hour
    datos["timestamp"] = datos["date_dt"]

print("\nHoras detectadas en el conjunto de datos:")
print(sorted(datos["hora"].dropna().unique()))


# Transformamr el dataset de formato largo a ancho (una columna por contaminante)
tabla = datos.pivot_table(
    index=["timestamp", "id_station"],
    columns="id_parameter",
    values="valor",
    aggfunc="mean",
).reset_index()

tabla.columns.name = None

print("\nNuevas dimensiones tras el pivoteo:", tabla.shape)
print("Columnas disponibles:", tabla.columns.tolist())
display(tabla.head())

# Registro de datos faltantes por parámetro
print("\nConteo de valores nulos por variable:")
print(tabla.isna().sum())


# Evaluación de relación lineal entre Monóxido de Carbono y Óxidos de Nitrógeno
co_nox = tabla[["CO", "NOX"]].dropna()
cov_co_nox = co_nox["CO"].cov(co_nox["NOX"])
corr_co_nox = co_nox["CO"].corr(co_nox["NOX"])

print(f"\nMuestras completas CO-NOX evaluadas: {len(co_nox)}")
print(f"Covarianza CO-NOX: {cov_co_nox:.4f}")
print(f"Correlación CO-NOX: {corr_co_nox:.4f}")

# Lista global de parámetros
contaminantes = [
    c
    for c in [
        "CO",
        "NO",
        "NO2",
        "NOX",
        "O3",
        "PM10",
        "PM2.5",
        "PMCO",
        "SO2",
    ]
    if c in tabla.columns
]

# Descomposición espectral (Eigenvalores) comparativa
corr_faltantes = tabla[contaminantes].corr()
eigen_faltantes = np.sort(np.linalg.eigvalsh(corr_faltantes.values))[::-1]

tabla_completa = tabla[contaminantes].dropna()
corr_limpia = tabla_completa.corr()
eigen_limpios = np.sort(np.linalg.eigvalsh(corr_limpia.values))[::-1]

comparativa_eigen = pd.DataFrame(
    {"Con Faltantes": eigen_faltantes, "Sin Faltantes": eigen_limpios}
)
print("\nComparación de Eigenvalores de la Matriz de Correlación:")
display(comparativa_eigen)


target = "PM2.5"
predictores_todos = [p for p in contaminantes if p != target]

# Modelo 1: Todos los contaminantes
df_mod_todos = tabla[[target] + predictores_todos].dropna()
X_t = sm.add_constant(df_mod_todos[predictores_todos])
y_t = df_mod_todos[target]

modelo_todos = sm.OLS(y_t, X_t).fit()
print("\nSummary Modelo Global (Todos los contaminantes):")
print(f"R² alcanzado: {modelo_todos.rsquared:.4f}")

# Modelo 2: Exclusivamente gases
gases = [g for g in ["CO", "NO", "NO2", "NOX", "O3", "SO2"] if g in tabla.columns]
df_mod_gases = tabla[[target] + gases].dropna()
X_g = sm.add_constant(df_mod_gases[gases])
y_g = df_mod_gases[target]

modelo_gases = sm.OLS(y_g, X_g).fit()
print(f"\nR² Modelo Restringido (Solo gases): {modelo_gases.rsquared:.4f}")

# Factores de Inflación de Varianza (VIF) para Gases
vif_gases = pd.DataFrame(
    {
        "Variable": gases,
        "VIF": [
            variance_inflation_factor(X_g.iloc[:, 1:].values, i)
            for i in range(len(gases))
        ],
    }
)
print("\nEvaluación de Multicolinealidad (VIF - Gases):")
display(vif_gases)

predictores_h = ["CO", "NO", "NO2", "O3", "SO2"]
base_h = tabla[["timestamp", "id_station", "PM2.5"] + predictores_h].dropna()
base_h["semana"] = base_h["timestamp"].dt.to_period("W")

# Seleccionamos la semana/estación con mayor presencia de registros completos
estacion_top, semana_top = (
    base_h.groupby(["id_station", "semana"]).size().idxmax()
)
semana_sel = base_h[
    (base_h["id_station"] == estacion_top)
    & (base_h["semana"] == semana_top)
].sort_values("timestamp")

X_sem = np.column_stack(
    [np.ones(len(semana_sel)), semana_sel[predictores_h].to_numpy()]
)
n_sem, p_sem = X_sem.shape

# Construcción explícita de H = X * (X'X)^(-1) * X'
XtX_inv_sem = np.linalg.inv(X_sem.T @ X_sem)
H = X_sem @ XtX_inv_sem @ X_sem.T

print(f"\nVerificación Matriz H ({n_sem}x{n_sem}):")
print(f"- Idempotencia (H² = H): {np.allclose(H @ H, H)}")
print(f"- Traza igual a parámetros p ({p_sem}): {np.isclose(np.trace(H), p_sem)}")

datos_lev = tabla[["timestamp", "id_station", "PM2.5"] + predictores_h].dropna().reset_index(drop=True)
X_all = np.column_stack([np.ones(len(datos_lev)), datos_lev[predictores_h].to_numpy()])

# Obtención eficiente de los elementos diagonales h_ii sin generar H completa
XtX_inv_all = np.linalg.inv(X_all.T @ X_all)
datos_lev["leverage"] = np.einsum("ij,jk,ik->i", X_all, XtX_inv_all, X_all)

# Cálculo de la Distancia de Cook
modelo_cook = sm.OLS(datos_lev["PM2.5"], X_all).fit()
datos_lev["cook_distance"] = modelo_cook.get_influence().cooks_distance[0]

# Comparación de los Top 10
top10_lev = datos_lev.sort_values("leverage", ascending=False).head(10)
top10_cook = datos_lev.sort_values("cook_distance", ascending=False).head(10)

print("\n--- Top 10 Observaciones con Mayor Leverage ---")
display(top10_lev[["timestamp", "id_station", "PM2.5", "leverage"]])

print("\n--- Top 10 Observaciones con Mayor Distancia de Cook ---")
display(
    top10_cook[
        ["timestamp", "id_station", "PM2.5", "leverage", "cook_distance"]
    ]
)

# Gráfica de Dispersión
plt.figure(figsize=(8, 4.5))
plt.scatter(
    datos_lev["leverage"],
    datos_lev["cook_distance"],
    alpha=0.4,
    c="#1f77b4",
    edgecolors="none",
)
plt.axvline(
    2 * X_all.shape[1] / len(datos_lev),
    color="red",
    linestyle="--",
    label="Umbral crítico de Leverage",
)
plt.xlabel("Apalancamiento (Leverage)")
plt.ylabel("Distancia de Cook")
plt.title("Diagnóstico de Muestras: Distancia de Cook vs Leverage")
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()